In [1]:
# ── Cell 1: Config ────────────────────────────────────────────────────────────
MODEL_NAME     = "numind/NuExtract-2.0-2B"  # or local path e.g. /raid/dmdsouza/models/NuExtract-2.0-2B
USE_API        = False   # True = HF Serverless API, False = local/GPU
HF_TOKEN       = ""      # only needed if USE_API=True
SAMPLE_SIZE    = 5       # None = all 5010 rows, int = subset e.g. 50
MAX_NEW_TOKENS = 4000
RESULTS_DIR    = "evaluation/results"

import json
import os
from pathlib import Path

# On the remote GPU machine (/raid/dmdsouza), set HF_HOME so model weights
# download to RAID instead of filling the home directory quota.
RAID_CACHE = Path("/raid/dmdsouza/.cache/huggingface")
if RAID_CACHE.parent.parent.exists():  # /raid/dmdsouza exists → we're on the remote machine
    os.environ["HF_HOME"] = str(RAID_CACHE)
    print(f"HF_HOME set to {RAID_CACHE}")

# Resolve paths relative to the project root (one level up from notebooks/)
PROJECT_ROOT = Path(__file__).parent.parent if "__file__" in dir() else Path().resolve().parent
DATA_PATH    = PROJECT_ROOT / "data/synthesized_data/nuextract_prompts.parquet"
RESULTS_DIR  = PROJECT_ROOT / RESULTS_DIR

print(f"Project root: {PROJECT_ROOT}")
print(f"Data path:    {DATA_PATH}")

HF_HOME set to /raid/dmdsouza/.cache/huggingface
Project root: /raid/dmdsouza/exactus
Data path:    /raid/dmdsouza/exactus/data/synthesized_data/nuextract_prompts.parquet


In [2]:
# ── Cell 2: Install & imports ──────────────────────────────────────────────────
# On Colab, uncomment the pip install line.
# !pip install -q transformers torch pyarrow huggingface_hub tqdm qwen-vl-utils

import json
import os
import csv
from pathlib import Path
from tqdm.auto import tqdm
import pyarrow.parquet as pq

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Imports OK")

Imports OK


In [3]:
# ── Cell 3: Load dataset ───────────────────────────────────────────────────────
table = pq.read_table(DATA_PATH)
data  = table.to_pydict()

texts     = data["text"]
templates = data["template"]
targets   = data["target"]

if SAMPLE_SIZE:
    texts     = texts[:SAMPLE_SIZE]
    templates = templates[:SAMPLE_SIZE]
    targets   = targets[:SAMPLE_SIZE]

print(f"Loaded {len(texts)} rows from {DATA_PATH}")

Loaded 5 rows from /raid/dmdsouza/exactus/data/synthesized_data/nuextract_prompts.parquet


In [4]:
# ── Cell 4: Load model or API client ──────────────────────────────────────────
if USE_API:
    from huggingface_hub import InferenceClient
    client = InferenceClient(model=MODEL_NAME, token=HF_TOKEN or None)
    print(f"Using HF Inference API: {MODEL_NAME}")
else:
    import torch
    from transformers import AutoProcessor, AutoModelForVision2Seq
    from qwen_vl_utils import process_vision_info

    processor = AutoProcessor.from_pretrained(MODEL_NAME)
    model = AutoModelForVision2Seq.from_pretrained(
        MODEL_NAME,
        dtype="auto",
        device_map="auto",
    )
    model.eval()
    print(f"Loaded {MODEL_NAME} on {next(model.parameters()).device}")

/raid/dmdsouza/exactus/.venv/lib64/python3.9/site-packages/networkx/utils/backends.py:135: RuntimeWarning: networkx backend defined more than once: nx-loopback
  backends.update(_get_backends("networkx.backends"))


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


/raid/dmdsouza/exactus/.venv/lib64/python3.9/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


Loaded numind/NuExtract-2.0-2B on cuda:0


In [5]:
# ── Cell 5: Run inference ──────────────────────────────────────────────────────

def predict_NuExtract(model, processor, texts, templates, batch_size=1):
    outputs = []
    for i in range(0, len(texts), batch_size):
        batch_texts     = texts[i:i + batch_size]
        batch_templates = templates[i:i + batch_size]

        formatted = []
        for text, template in zip(batch_texts, batch_templates):
            messages = [{"role": "user", "content": text}]
            formatted.append(
                processor.tokenizer.apply_chat_template(
                    messages,
                    template=json.dumps(json.loads(template), indent=4),
                    tokenize=False,
                    add_generation_prompt=True,
                )
            )

        image_inputs = process_vision_info([{"role": "user", "content": t} for t in batch_texts])[0]
        inputs = processor(
            text=formatted,
            images=image_inputs,
            padding=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                num_beams=1,
                repetition_penalty=1.1,
                pad_token_id=processor.tokenizer.eos_token_id,
            )
        trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
        outputs += processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    return outputs


def run_inference_api(text, template):
    messages = [{"role": "user", "content": text}]
    response = client.chat_completion(messages=messages, max_tokens=MAX_NEW_TOKENS, temperature=0.0)
    return response.choices[0].message.content


raw_outputs = []
if USE_API:
    for text, template in tqdm(zip(texts, templates), total=len(texts), desc="Running inference (API)"):
        raw_outputs.append(run_inference_api(text, template))
else:
    raw_outputs = predict_NuExtract(model, processor, list(texts), list(templates), batch_size=1)

print(f"Inference complete. Sample output:\n{raw_outputs[0][:300]}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Inference complete. Sample output:
{"Concept": [], "Government Entity": [], "Government Institution": [], "Relations": {"executes": [{"subject": "the law", "object": null}], "is_part_of": []}}


In [6]:
# ── Cell 6: Accumulate TP/FP/FN counts ────────────────────────────────────────
# Uses micro-F1 methodology: accumulate raw counts across ALL rows,
# compute F1 once at the end. Also tracks per entity-type and per relation-type.
# Matching is case-insensitive; entity type is included in the match.

from collections import defaultdict
import json

def parse_output(raw: str):
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        start, end = raw.find("{"), raw.rfind("}") + 1
        if start != -1 and end > start:
            try:
                return json.loads(raw[start:end])
            except json.JSONDecodeError:
                pass
    return None


# Global micro-F1 accumulators
entity_counts   = {"tp": 0, "fp": 0, "fn": 0}
relation_counts = {"tp": 0, "fp": 0, "fn": 0}

# Per-class accumulators
per_entity_type   = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})
per_relation_type = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})

results = []

for i, (raw, target_str, text) in enumerate(zip(raw_outputs, targets, texts)):
    target_parsed = json.loads(target_str)
    pred_parsed   = parse_output(raw)
    parse_error   = pred_parsed is None
    exact_match   = False
    row_entity_correct = row_relation_correct = True

    if not parse_error:
        # ── Entity matching (per entity-type, case-insensitive name) ──────────
        for entity_type, gt_vals in target_parsed.items():
            if entity_type == "Relations" or not isinstance(gt_vals, list):
                continue
            gt_names   = {n.lower() for n in gt_vals if isinstance(n, str)}
            pred_names = {n.lower() for n in pred_parsed.get(entity_type, []) if isinstance(n, str)}
            tp = len(gt_names & pred_names)
            fp = len(pred_names - gt_names)
            fn = len(gt_names - pred_names)
            entity_counts["tp"] += tp
            entity_counts["fp"] += fp
            entity_counts["fn"] += fn
            per_entity_type[entity_type]["tp"] += tp
            per_entity_type[entity_type]["fp"] += fp
            per_entity_type[entity_type]["fn"] += fn
            if fp or fn:
                row_entity_correct = False

        # ── Relation matching (per predicate, case-insensitive subject+object) ─
        gt_rels   = target_parsed.get("Relations", {})
        pred_rels = pred_parsed.get("Relations", {}) if isinstance(pred_parsed, dict) else {}
        all_predicates = set(gt_rels) | set(pred_rels)

        for predicate in all_predicates:
            gt_pairs   = {((r.get("subject") or "").lower(), (r.get("object") or "").lower())
                          for r in gt_rels.get(predicate, []) if isinstance(r, dict)}
            pred_pairs = {((r.get("subject") or "").lower(), (r.get("object") or "").lower())
                          for r in pred_rels.get(predicate, []) if isinstance(r, dict)}
            tp = len(gt_pairs & pred_pairs)
            fp = len(pred_pairs - gt_pairs)
            fn = len(gt_pairs - pred_pairs)
            relation_counts["tp"] += tp
            relation_counts["fp"] += fp
            relation_counts["fn"] += fn
            per_relation_type[predicate]["tp"] += tp
            per_relation_type[predicate]["fp"] += fp
            per_relation_type[predicate]["fn"] += fn
            if fp or fn:
                row_relation_correct = False

        exact_match = row_entity_correct and row_relation_correct

    results.append({
        "index":        i,
        "text":         text,
        "target":       target_str,
        "model_output": raw,
        "parse_error":  parse_error,
        "exact_match":  exact_match,
    })

print(f"Processed {len(results)} rows.")

Processed 5 rows.


In [7]:
# ── Cell 7: Aggregate micro-F1 + per-class breakdown ──────────────────────────

def compute_f1(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return {"precision": round(p, 4), "recall": round(r, 4), "f1": round(f, 4)}

total   = len(results)
n_exact = sum(r["exact_match"] for r in results)
n_parse_errors = sum(r["parse_error"] for r in results)

summary = {
    "model":                MODEL_NAME,
    "total_rows":           total,
    "sample_size":          SAMPLE_SIZE,
    "schema_compliance":    round((total - n_parse_errors) / total, 4) if total else 0,
    "exact_match_accuracy": round(n_exact / total, 4) if total else 0,
    "entity_micro":         compute_f1(**entity_counts),
    "relation_micro":       compute_f1(**relation_counts),
    "per_entity_type":      {k: compute_f1(**v) for k, v in sorted(per_entity_type.items())},
    "per_relation_type":    {k: compute_f1(**v) for k, v in sorted(per_relation_type.items())},
}

print(f"Schema compliance:    {summary['schema_compliance']:.1%}")
print(f"Exact match accuracy: {summary['exact_match_accuracy']:.1%}")
print(f"Entity micro-F1:      {summary['entity_micro']['f1']}")
print(f"Relation micro-F1:    {summary['relation_micro']['f1']}")

Schema compliance:    100.0%
Exact match accuracy: 0.0%
Entity micro-F1:      0.4
Relation micro-F1:    0.0


In [8]:
# ── Cell 8: Save results ───────────────────────────────────────────────────────

model_tag    = MODEL_NAME.split("/")[-1].lower().replace("-", "_")
summary_path = RESULTS_DIR / f"{model_tag}_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print(f"Saved summary -> {summary_path}")

errors      = [r for r in results if not r["exact_match"]]
errors_path = RESULTS_DIR / f"{model_tag}_errors.csv"
if errors:
    fieldnames = ["index", "text", "target", "model_output", "parse_error"]
    with open(errors_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(errors)
    print(f"Saved {len(errors)} error rows -> {errors_path}")
else:
    print("No errors — perfect score!")

Saved summary -> /raid/dmdsouza/exactus/evaluation/results/nuextract_2.0_2b_summary.json
Saved 5 error rows -> /raid/dmdsouza/exactus/evaluation/results/nuextract_2.0_2b_errors.csv
